# 01 — Topic Segmentation Visualizer (TextTiling, real embeddings)

This notebook runs EgoVault's actual segmentation engine — `tools.text.segment.segment_chunks` —
on two corpora, using the **real embedding provider** (Ollama `nomic-embed-text`, 768 dims,
via `infrastructure/embedding_provider.py`). No mock vectors: if Ollama is not running,
the next cell fails loud with setup instructions instead of silently degrading.

1. **Corpus A** — a synthetic 3-topic sample (Quantum Computing / Cognitive Neuroscience / Game Theory), chunked small for legibility.
2. **Corpus B** — the first 20 pages of the real *Marcus Aurelius: Meditations* corpus, chunked with production settings (`config/system.yaml`).

For each corpus we expose the three hidden signals that drive every boundary decision:
consecutive cosine similarity, the peak-relative **valley depth score**, and the adaptive
**statistical threshold** (`mean + k · std`, `k = note_segmentation.sensitivity_k`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import load_settings
from infrastructure.context import build_context
from notebooks._lib.embedding_cache import probe_provider
from notebooks.demo_segmentation_visualizer import SAMPLE_TEXT, run_corpus

settings = load_settings()
ctx = build_context(settings)
probe_provider(ctx)
print("EgoVault Segmentation Visualizer ready — embedding provider reachable.")

## 1. Corpus A — Synthetic 3-Topic Sample

Chunk size is overridden to 100/20 words for this cell only (the production 300-word chunk
size would collapse each ~180-word section into a single chunk, hiding the boundary signal
entirely). This is a demo-legibility override, not a production default — Corpus B below
uses the real `config/system.yaml` values untouched.

In [ ]:
chunks_a, candidates_a = run_corpus(
    "Corpus A: Synthetic 3-Topic Sample (clean boundaries)",
    SAMPLE_TEXT, ctx, "segmentation_synthetic_signal.png",
    chunking_override=(100, 20),
)

**Reading the plot:** with real semantic embeddings, this short synthetic sample may or
may not cross the calibrated `sensitivity_k=1.2` threshold — that's the point. The previous
version of this notebook used hash-based bag-of-words vectors, which produced sharper
artificial boundaries than real embeddings do on short, clean text. This is the honest
behavior of the engine, not a bug.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / "notebooks" / "assets" / "segmentation_synthetic_signal.png")))

## 2. Corpus B — Marcus Aurelius Meditations (real, noisy)

Same pipeline, production chunking config (`size=300`, `overlap=40`), first 20 pages of the
real PDF. This is the harder, noisier case: 128-page prose with no clean section markers.

In [ ]:
import pypdf

pdf_file = PROJECT_ROOT / "notebooks" / "assets" / "Marcus-Aurelius-Meditations.pdf"
reader = pypdf.PdfReader(str(pdf_file))
pages_text = [page.extract_text() for page in reader.pages[:20]]
real_text = "\n\n".join(filter(None, pages_text))

chunks_b, candidates_b = run_corpus(
    "Corpus B: Marcus Aurelius Meditations, first 20 pages (real, noisy)",
    real_text, ctx, "segmentation_real_signal.png",
)

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "notebooks" / "assets" / "segmentation_real_signal.png")))

## 3. Takeaway

The valley-depth + adaptive-threshold mechanism is the same code path production ingestion
uses (`tools/text/segment.py:segment_chunks`). Real embeddings are noisier and less
separable than the bag-of-words mock previously used here — segmentation sensitivity
(`note_segmentation.sensitivity_k` in `config/system.yaml`) should be calibrated against
real embedding output, not against a mock that overstates topic separability.